# Spike 0.6c.2 — oscillating NACA-0012 SU2 solver validation

**Runs in parallel with the Phase-4 / Phase-5 notebooks — on a *separate* Colab runtime.** This is the **SU2 side** of the cross-solver gate: it validates SU2's unsteady laminar physics on the canonical pitching-airfoil case (NACA 0012, Re ~ 40k, reduced frequency *k* ~ 0.55, ±10° about the quarter chord) so we can trust the Tier-1 unsteady numerics.

Per run: build the airfoil mesh → render the pitching cfg → run unsteady SU2 → reduce the lift/drag history to **C_L,max**, **C_d,mean**, and the **C_L–α hysteresis-loop area**.

**CPU only** (2D SU2 — no GPU). Thin orchestration; logic lives in tested `fanopt.cfd.{airfoil_mesh,naca_benchmark}`.

> **The PASS/FAIL comparison is a separate step.** It needs (a) PyFR p=3 on the G4 GPU (Round-9 HIGH-11) and (b) **published dynamic-stall reference numbers** — which are *not* invented here. This notebook produces the SU2-side observables that feed that comparison.

## 1. Repo + Python/native deps

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# One long serial SU2 run; pin BLAS single-threaded so nothing oversubscribes.
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    REPO_DIR = Path("/content/fan-optimization")
    BRANCH = "main"
    REPO_URL = "https://github.com/clingergab/fan-optimization.git"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    subprocess.run(
        "apt-get install -qq -y libglu1-mesa libxrender1 libxcursor1 "
        "libxft2 libxinerama1 unzip".split(),
        check=True,
    )
    # NACA chain is lean: gmsh + scipy + jinja2 + pyyaml (no cadquery / botorch).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "gmsh", "scipy", "jinja2", "pyyaml", "tqdm", "matplotlib"],
        check=True,
    )
else:
    REPO_DIR = Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "pyproject.toml").exists():
        REPO_DIR = REPO_DIR.parent

for p in (REPO_DIR / "src", REPO_DIR / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

assert (REPO_DIR / "src" / "fanopt" / "cfd" / "naca_benchmark.py").exists(), \
    "naca_benchmark missing — is the branch pushed?"

_missing = [m for m in ("gmsh", "scipy", "jinja2", "yaml")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise ModuleNotFoundError(
        f"Missing {_missing} in this kernel:\n  {sys.executable}\n"
        "Local: pick the kernel that has gmsh; Colab: re-run this cell (it installs them)."
    )
print("repo:", REPO_DIR, "| colab:", IN_COLAB)

## 2. SU2 — install (Colab) or locate (local)

In [ ]:
import importlib.util
import os
import subprocess
import urllib.request
from pathlib import Path

if importlib.util.find_spec("google.colab") is not None:
    SU2_VERSION = "8.0.1"
    SU2_DIR = Path("/content/su2")
    if not any(SU2_DIR.rglob("SU2_CFD")):
        url = (
            f"https://github.com/su2code/SU2/releases/download/"
            f"v{SU2_VERSION}/SU2-v{SU2_VERSION}-linux64.zip"
        )
        print("[su2] downloading", url)
        urllib.request.urlretrieve(url, "/tmp/su2.zip")
        SU2_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", "/tmp/su2.zip", "-d", str(SU2_DIR)], check=True)
    cands = [p for p in SU2_DIR.rglob("SU2_CFD") if p.is_file() and not p.is_symlink()]
    assert cands, "SU2_CFD not found after extract"
    bindir = cands[0].parent
    for p in bindir.iterdir():
        if p.is_file():
            os.chmod(p, 0o755)
else:
    bindir = Path.home() / "su2-local" / "extracted" / "bin"
    assert (bindir / "SU2_CFD").exists(), f"SU2_CFD not at {bindir}"

os.environ["SU2_RUN"] = str(bindir)
os.environ["PATH"] = str(bindir) + os.pathsep + os.environ.get("PATH", "")
print("SU2_RUN:", os.environ["SU2_RUN"])

## 3. Settings — physical case + fidelity

`QUICK = True` is a coarse smoke run (a few minutes) that only proves the pipeline turns over — **its numbers are unconverged and not physically meaningful.** Set `QUICK = False` for the resolved production run whose C_L / C_d you can actually compare to reference data (fine mesh, `inner_iter = 100`, 5 full cycles — expect tens of minutes on a Colab CPU).

In [ ]:
from pathlib import Path

QUICK = False   # False = resolved production run; True = fast pipeline smoke only

# --- Physical case (the literature reports it this way) ---------------------
REYNOLDS_NUMBER = 40000.0
REDUCED_FREQUENCY_K = 0.55
PITCH_AMPLITUDE_DEG = 10.0

# --- Fidelity ---------------------------------------------------------------
if QUICK:
    N_CYCLES, STEPS_PER_CYCLE, INNER_ITER = 2, 20, 5
    N_AIRFOIL_POINTS, WALL_MESH_SIZE_M, FARFIELD_MESH_SIZE_M = 60, 0.02, 1.2
else:
    N_CYCLES, STEPS_PER_CYCLE, INNER_ITER = 5, 200, 100
    N_AIRFOIL_POINTS, WALL_MESH_SIZE_M, FARFIELD_MESH_SIZE_M = 200, 0.004, 0.5

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/fanopt/spike_0_6c_naca")
else:
    OUT_DIR = REPO_DIR / "data" / "spike_0_6c" / "naca_nb"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"QUICK={QUICK} | Re={REYNOLDS_NUMBER:.0f} k={REDUCED_FREQUENCY_K} "
      f"+/-{PITCH_AMPLITUDE_DEG}deg | cycles={N_CYCLES} steps/cyc={STEPS_PER_CYCLE} "
      f"inner={INNER_ITER}\nout: {OUT_DIR}")

## 4. Run the benchmark

In [ ]:
import json
from dataclasses import asdict

from fanopt.cfd.airfoil_mesh import AirfoilMeshParams
from fanopt.cfd.naca_benchmark import BenchmarkConfig, run_benchmark

cfg = BenchmarkConfig(
    reynolds_number=REYNOLDS_NUMBER,
    reduced_frequency_k=REDUCED_FREQUENCY_K,
    pitch_amplitude_deg=PITCH_AMPLITUDE_DEG,
    n_cycles=N_CYCLES,
    steps_per_cycle=STEPS_PER_CYCLE,
    inner_iter=INNER_ITER,
    n_airfoil_points=N_AIRFOIL_POINTS,
    mesh_params=AirfoilMeshParams(
        wall_mesh_size_m=WALL_MESH_SIZE_M, farfield_mesh_size_m=FARFIELD_MESH_SIZE_M
    ),
)
print(f"U={cfg.freestream_velocity_ms:.2f} m/s  omega={cfg.pitching_omega_rad_s:.3f} rad/s  "
      f"dt={cfg.time_step_s:.5f}s  time_iter={cfg.time_iter}  (this runs SU2 now...)")

metrics = run_benchmark(cfg, OUT_DIR, su2_bin=None)
(OUT_DIR / "metrics.json").write_text(json.dumps(asdict(metrics), indent=2))

print("\nC_L,max        =", round(metrics.c_l_max, 4), "at alpha =",
      round(metrics.alpha_at_cl_max_deg, 2), "deg")
print("C_d,mean       =", round(metrics.c_d_mean, 5))
print("hysteresis area=", round(metrics.hysteresis_area, 5), "(rad*C_L)")
print("cycles used    =", metrics.n_cycles_used)
if QUICK:
    print("\n[QUICK] numbers are unconverged — set QUICK=False for a comparable run.")

## 5. C_L–α hysteresis loop + time histories

The dynamic-stall signature. The **loop area** and **C_L,max** are what get compared to PyFR and published data once reference numbers are in hand.

In [ ]:
import matplotlib.pyplot as plt

from fanopt.cfd.naca_benchmark import pitch_angle_series
from fanopt.cfd.parsers import parse_su2_unsteady_force_series

hist = sorted(OUT_DIR.glob("history*.csv"), key=lambda p: p.stat().st_mtime)[-1]
cl = parse_su2_unsteady_force_series(hist, force_candidates=("CL",))
cd = parse_su2_unsteady_force_series(hist, force_candidates=("CD",))
import numpy as np
alpha_deg = np.degrees(pitch_angle_series(cfg))
n = min(cl.size, cd.size, alpha_deg.size)
cl, cd, alpha_deg = cl[:n], cd[:n], alpha_deg[:n]

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.2))
a0.plot(alpha_deg, cl, lw=1.2)
a0.set_xlabel("angle of attack alpha (deg)")
a0.set_ylabel("C_L")
a0.set_title(f"C_L-alpha hysteresis loop  (C_L,max={metrics.c_l_max:.3f})")
a0.grid(alpha=0.3)

t = np.arange(n) * cfg.time_step_s
a1.plot(t, cl, label="C_L", lw=1.0)
a1.plot(t, cd, label="C_D", lw=1.0)
a1.set_xlabel("time (s)")
a1.set_ylabel("coefficient")
a1.set_title("Force history")
a1.legend()
a1.grid(alpha=0.3)
fig.tight_layout()
plt.show()